## Week 2 Day 2 - Orchestration

Our first Agentic Framework project!!

### Part 1: Email Setup

### Part 2: Orchestrating by code

### Part 3: Orchestrating by LLMs

- 3a: via Tools
- 3b: via Handoffs

## Part 1: Email Setup

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">IMPORTANT PLEASE READ - Sending Emails</h2>
            <span style="color:#ff7800;">We are going to write an agent that sends emails. The best way to do this is with an email provider
            like SendGrid or Resend. But this requires a fair amount of heavy-lifting: you need to send from a proper email host where you own the domain and you can prove it with DNS records. It's quite a hassle.<br/><br/>
            So we're going to do it the free & simple way: using SMTP server configuration to send direct from your email. It's an easy setup, but it's not as powerful (eg you can't receive emails). If you'd like to take this further, use SendGrid or Resend instead..<br/><br/>
            And actually the email send is optional. We're doing this to demonstrate an Agent sending emails. The important part is the Agent work, not the email sending. Feel free to swap out the function for a Pushover push if you'd prefer.
            </span>
        </td>
    </tr>
</table>

### Ufo: decided not to use the send email option, so will comment the code blocks about setting up email server. Will just use the Pushover option to focus on the Agent orchestration learning.

## Setting up to send emails from your SMTP server

### STEP 1: Determining your SMTP Server

Either google or ask ChatGPT / Claude for the SMTP server for your email. Here are some common ones. Some email providers might not have SMTP servers enabled (eg Microsoft 365 for work/school).

Google: smtp.gmail.com  
Outlook.com / Hotmail / Live: smtp-mail.outlook.com  
Microsoft 365: smtp.office365.com  
iCloud Mail: smtp.mail.me.com  

Add to your .env file:  

`EMAIL_SMTP_SERVER=xxxx`

### STEP 2: Obtain an app specific password

Google how to do this for your email provider. For gmail, you need to have 2-step verification on. Then visit this page:

https://myaccount.google.com/apppasswords

Give it any name; copy the password and add it to your .env file, removing the spaces that it adds. (It should be 16 characters, no spaces.)

`EMAIL_APP_PASSWORD=xxxx`

### STEP 3: Add in your email address:

`EMAIL_ADDRESS=xxx`

Remember to Save the .env file!

In [18]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import smtplib
# from email.message import EmailMessage
load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"


In [19]:
# EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
# EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
# EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

# if EMAIL_ADDRESS:
#     print("Email address is set")
# else:
#     print("Email address is not set")

# if EMAIL_SMTP_SERVER:
#     print("SMTP server is set")
# else:
#     print("SMTP server is not set")

# if EMAIL_APP_PASSWORD:
#     print("App password is set")
# else:
#     print("App password is not set")

# USE_EMAIL = EMAIL_ADDRESS and EMAIL_SMTP_SERVER and EMAIL_APP_PASSWORD

# if USE_EMAIL:
#     print("Email is set up and we will try using it")
# else:
#     print("Email is not set up; we will send push notifications instead")

In [20]:
# # Here we go

# def send_email(subject, text_body, html_body):
#     msg = EmailMessage()
#     msg["From"] = EMAIL_ADDRESS
#     msg["To"] = EMAIL_ADDRESS
#     msg["Subject"] = subject
#     msg.set_content(text_body)
#     msg.add_alternative(html_body, subtype="html")

#     with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
#         server.starttls()
#         server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
#         server.send_message(msg)

In [21]:
# send_email("Testing testing 123", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

### If this didn't work, then uncomment the below so that we don't use emails

In [22]:
USE_EMAIL = False

### Our fallback strategy - send a push

In [23]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [24]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [25]:
# This is the function that we will give to our Agent to send messages.
def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        # push(f"Subject: {subject}\n\n{text_body}")
        # print("Message is succesfully sent. Our Agents worked!") # if you don't want to use Push, just change this line to write to a local file or just print. What we want to focus on is the Agent orchestration part.

SyntaxError: incomplete input (1007907629.py, line 7)

### OK Now everything should work!

In [ ]:
send_message("Big news", "Communications are a go!", "<html><body>Communications are a <strong>go!</strong></body></html>")

Push: Subject: Big news

Communications are a go!


## Agent Orchestration

There are 2 models for Agent Orchestration; by code and by LLMs.

By code: more predictable and deterministic.

By LLMs: more powerful.

An excellent write-up is here:

https://openai.github.io/openai-agents-python/multi_agent/

We will start with by Code.

## Part 2: Orchestrating by Code

Ufo's learning notes about the two methods of ochestration:
* Ochestrating by code is the simplest one to explain. It basically means you will make multiple calls to runner.run(), multiple calls to send a prompt to an LLM, and then based on the repsonse from it, you construct the next prompt to the LLM, the next runner.run(). It's a manual process - it's similar to calling an LLM, taking the output and parsing it into another LLM input. It's the most simple, reliable, and bullet-proof way of doing it. 
* Ochestrating with Agent means you only make one runner.run() call, but you equip that Agent with some tools. When you implement the tools, that tools involve calling another Agent. You equip the LLM with tools, and those tools send messages to more LLMs. This method gives the LLM more automomous and more freedom to decide how it wants to collaborate with other Agents based on its own reasoning and the tokens it generates. You're allowing it to ochestrate the workflow. It's considered ochestration by LLM. 

In [ ]:
intro = """
You are a sales agent working for ComplAI, 
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [ ]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL_NAME)


In [ ]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Making SOC 2 readiness simpler and more defensible

Hi [First Name],

I’m reaching out from ComplAI. We help teams prepare for SOC 2 with a SaaS platform that uses AI to reduce the manual work involved in compliance and audit readiness.

For many organizations, the challenge is not just passing an audit — it’s maintaining clear, defensible evidence and keeping compliance work from becoming a distraction for the engineering and operations teams. That is where ComplAI helps: by streamlining evidence collection, tracking controls, and making audit preparation more manageable.

If SOC 2 is on your roadmap, I’d welcome the chance to show you how teams are using ComplAI to get ready faster and with more confidence.

Would you be open to a brief conversation next week?

Best regards,  
[Your Name]  
ComplAI

In [ ]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(  # using async.io to "kinda" run in parallel. async.io runs one while waiting for the other.
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Subject: Simplify SOC 2 readiness without slowing your team down

Hi [First Name],

I’m reaching out from ComplAI. We help companies prepare for SOC 2 audits faster and with far less manual effort by using AI to streamline evidence collection, control tracking, and audit readiness.

For many teams, SOC 2 becomes a drain on engineering, security, and operations resources. ComplAI is designed to reduce that burden by giving you a clearer path to compliance, so your team can stay focused on building.

If SOC 2 is on your roadmap this quarter, I’d welcome the chance to show you how teams are using ComplAI to get audit-ready with more confidence and less overhead.

Would you be open to a brief conversation next week?

Best,  
[Your Name]  
ComplAI


Subject: SOC 2 compliance, minus the soul-crushing spreadsheet olympics

Hi [First Name],

Hope your week’s going well and your calendar is being kind to you.

I’m reaching out because a lot of teams we speak with are trying to get SOC 2 audit-r

In [ ]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

sales_picker = Agent(name="Sales_picker", instructions=decision, model=MODEL_NAME)


In [ ]:
# have all four agents work together: 3 agents to write three email styles and 1 agent to make decision on which email style to use.
message = "Write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
Subject: SOC 2, but make it less terrifying

Hi [First Name],

Hope your week’s going well and your compliance stack isn’t currently held together by spreadsheets, hope, and caffeine.

I’m reaching out from ComplAI, where we help teams get SOC 2 audit-ready without turning the process into a full-time side quest. Our AI-powered platform helps you stay on top of controls, evidence collection, and audit prep — so you can spend less time chasing screenshots and more time building the company.

In short: fewer compliance headaches, fewer “quick questions” from auditors, and a much calmer path to SOC 2.

If it’s useful, I’d love to show you how teams like yours are using ComplAI to simplify the whole thing.

Would you be open to a quick 15-minute chat next week?

Best,  
[Your Name]  
ComplAI


Now go and check out the trace:

https://platform.openai.com/traces

### Now we will add a tool to the mix.

In [ ]:
#This is not a tool to call an agent, just a normal tool. 

@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

### This has automatically been converted into a tool, with the boilerplate json created

In [ ]:
# checking the json schema created by the @function_tool decorator
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [ ]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

require_tool = ModelSettings(tool_choice="required") # This is a feature of the framework. This settings help make sure the LLM use your tool. Without this, the model can be unreliable, even with the newer LLM models. You can include the requirement in the prompt but this feature forces the LLM to use the tool at least once. 

sales_sender = Agent(name="Sales Sender", instructions=decision, model=MODEL_NAME, tools=[send_email_tool], model_settings=require_tool)

In [ ]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)

    print(f"Final response:\n{response.final_output}")

Push: Subject: SOC 2 compliance, minus the soul-crushing spreadsheet chaos

Hi [First Name],

If SOC 2 prep has your team living in a permanent state of “where is that evidence again?”, ComplAI can help.

We built an AI-powered SaaS tool that helps teams stay on top of SOC 2 compliance and get audit-ready without the usual spreadsheet sprawl, document scavenger hunts, and late-night panic.

With ComplAI, you can:
- Organize evidence in one place
- Track compliance tasks without the chaos
- Prep for audits faster with AI support

Basically: less “we should really fix this soon,” more “oh wow, we’re actually ready.”

If it’s helpful, I’d be happy to show you how teams use ComplAI to make SOC 2 feel a lot less like a fire drill.

Open to a quick chat next week?

Best,
[Your Name]
ComplAI
Final response:
Done — I picked the second email and sent it.


### Did that work?!

See the traces for more! This is a great way to debug. Smaller models might require more time and experimentation.

https://platform.openai.com/traces

## Part 3: Orchestrating by LLMs

### 3a: via Tools

The simplest way to have 1 Agent choose to invoke another is by treating it as a tool call.

The OpenAI Agents SDK gives a very simple way to do this.

This works best when the flow is:

Agent A -> Agent B -> Agent A

And for the classic "Planning Agent" situation.

In [ ]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

# If you already have an agent in the OpenAI Agents SDK, you can convert it to a tool by using the as_tool() method.
# By converting it to a tool, that just means that it writes a boilerplate tool that calls that agent. It will take the tool_description as the task in the user prompt and pass it into the agent tool_name
# Basically, it's a wrapper that generates a tool from an agent and can be called using the same runner.run()

tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool1

FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_email_writer_1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x121228a40>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [ ]:
tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]

tools

[FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_email_writer_1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x121229b20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_email_writer_2', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_email_writer_2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 

## And now it's time for our Sales Manager - our planning agent

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=MODEL_NAME)


In [ ]:
# use this tool to draw the flow graphically
draw_graph(sales_manager)

ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

Push: Subject: SOC 2 readiness, without the drag

Hi [First Name],

If SOC 2 is on your roadmap, ComplAI can help you get audit-ready faster.

We automate the busywork around evidence collection, control tracking, and audit prep so your team can stay focused on building. Our AI-powered platform is designed to reduce manual effort and keep compliance moving.

If helpful, I’d be glad to show you how teams use ComplAI to cut down prep time and simplify the path to SOC 2.

Worth a quick conversation next week?

Best,
[Your Name]


## Remember to check the trace

https://platform.openai.com/traces

And then check your email!! Also look in your Junk / Spam folder - after all, this is basically a spam message..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Unreliable results?</h2>
            <span style="color:#ff7800;">This is par for the course with Agentic AI, and particularly with Orchestrating by LLM.
            To resolve it, you'll need to experiment and iterate with the prompts. Particularly with smaller models, more experiments
            may be needed to get reliable outcomes.
            </span>
        </td>
    </tr>
</table>

## Part 3: Orchestrating by LLMs

### 3a: via Handoffs

I am not a fan of handoffs. They seem very unreliable. They're not used consistently by other frameworks.

Behind the scenes, OpenAI Agents SDK has implemented these with Tools anyway.

### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

A -> B -> A

With handoffs, control passes across

A -> B

In [ ]:

instructions = """
You are a Sales Manager at ComplAI. You get your sales team to draft emails, then send them all to a sales picker.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Handoff to the sales sender to choose and send the best email.
"""

tools = [tool1, tool2, tool3]
handoffs = [sales_sender]

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, handoffs=handoffs, model=MODEL_NAME)


In [ ]:
draw_graph(sales_manager)

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

Note that handoffs can be unrealiable and a little bit frustrating. I needed to force the tool use otherwise this didn't work. If you don't get reliable behavior, try iterating on the prompts - or use a larger model. And enjoy the process; this is what Agentic AI is all about!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Try extending with some more orchestration - perhaps another iteration of the email to refine it. 
            Use Orchestrating via Code and Orchestrating via LLMs.  
            HARD CHALLENGE: switch to using a professional email provider like SendGrid or Resend. Then handle the user replying. 
            Then have the SDR respond to keep the conversation going!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>